# 📘 Notebook – Tabela 4

**Acesso aos serviços preventivos de saúde das mulheres com filhos**

PNS 2013 e 2019 — Mulheres com 25 anos ou mais

In [1]:
    # -------------------------------------------------
# Tabela 4
# População: Mulheres ≥ 25 anos com filhos
# Fonte: PNS 2013 e 2019
# Unidade: Percentual (%)
# -------------------------------------------------


import pandas as pd
import numpy as np
import sys
from pathlib import Path
from IPython.display import display

# Setup do ambiente
sys.path.append(str(Path('..').resolve()))
from service.pns_service import get_dataframe

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [2]:
# -------------------------------
# Carregamento dos dados (gold)
# -------------------------------
variables = [
    'tem_filhos_nascidos_vivos',
    'preventivo_fez',
    'mamografia_fez',
    'preventivo_cobertura',
    'mamografia_cobertura',
]

sources = ['2013', '2019']

df = get_dataframe(
    variables=variables,
    sources=sources,
    filters={
        'sexo': '2',
        'idade': {'operador': '>=', 'valor': 25}
    }
)


In [3]:
# -------------------------------
# Restrição: mulheres com filhos
# -------------------------------
df_filhos = df[df['tem_filhos_nascidos_vivos'] >= 1].copy()


In [4]:
# -------------------------------
# Funções auxiliares
# -------------------------------


# Atenção: estatísticas não ponderadas por peso amostral

def tabela_acesso(df, col):
    fez = (df[col].mean() * 100).round(2)
    nao_fez = (100 - fez).round(2)
    return pd.Series(
        [fez, nao_fez, 100.0],
        index=['Realizou', 'Não realizou', 'Total']
    )


def tabela_cobertura(df, col):
    dist = df[col].value_counts(normalize=True) * 100
    dist = dist.round(2)
    dist = dist.sort_index()
    dist.loc['Total'] = 100.0
    return dist


In [5]:
# -------------------------------
# Garantia de tipos numéricos
# -------------------------------
for col in ["preventivo_fez", "mamografia_fez"]:
    df_filhos[col] = (
        df_filhos[col]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .astype(float)
    )
assert col in df_filhos.columns

In [6]:
# -------------------------------
# Construção da Tabela 4 (por ano)
# -------------------------------
tabelas = {}

for ano in sorted(df_filhos["origem"].unique()):
    df_ano = df_filhos[df_filhos["origem"] == ano]

    acesso = pd.DataFrame({
        "Papanicolau": tabela_acesso(df_ano, "preventivo_fez"),
        "Mamografia": tabela_acesso(df_ano, "mamografia_fez")
    })

    cobertura = pd.DataFrame({
        "Papanicolau": tabela_cobertura(
            df_ano[df_ano["preventivo_fez"] == 1],
            "preventivo_cobertura"
        ),
        "Mamografia": tabela_cobertura(
            df_ano[df_ano["mamografia_fez"] == 1],
            "mamografia_cobertura"
        )
    })

    tabelas[ano] = {
        "Acesso": acesso,
        "Cobertura": cobertura
    }



In [7]:
# -------------------------------
# Exibição final (formato artigo)
# -------------------------------
for ano, blocos in tabelas.items():
    print(f"\nTabela 4 – Acesso aos serviços preventivos de saúde das mulheres com filhos (%), PNS {ano}")
    display(blocos['Acesso'].rename_axis(""))

    print(f"\nTabela 4 – Tipo de cobertura dos exames entre mulheres com filhos (%), PNS {ano}")
    display(blocos['Cobertura'].rename_axis(""))



Tabela 4 – Acesso aos serviços preventivos de saúde das mulheres com filhos (%), PNS 2013


,Papanicolau,Mamografia
,,
Realizou,92.13,29.18
Não realizou,7.87,70.82
Total,100.00,100.00



Tabela 4 – Tipo de cobertura dos exames entre mulheres com filhos (%), PNS 2013


,Papanicolau,Mamografia
,,
Pagou,14.28,13.02
Plano,23.05,38.83
SUS,62.67,48.16
Total,100.00,100.00



Tabela 4 – Acesso aos serviços preventivos de saúde das mulheres com filhos (%), PNS 2019


,Papanicolau,Mamografia
,,
Realizou,93.22,50.38
Não realizou,6.78,49.62
Total,100.00,100.00



Tabela 4 – Tipo de cobertura dos exames entre mulheres com filhos (%), PNS 2019


,Papanicolau,Mamografia
,,
Pagou,29.23,33.49
SUS,70.77,66.51
Total,100.00,100.00


In [11]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

def exportar_tabela4_final_pdf(tabelas, filename="Tabela_04_Final.pdf"):
    # Criar o arquivo PDF
    with PdfPages(filename) as pdf:
        # Configuração da página (Tamanho A4: 8.27 x 11.69 polegadas)
        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis('off')

        # 1. Cabeçalho Superior
        plt.text(0.5, 0.94, "Tabela 4 – Perfil de Acesso e Cobertura dos Exames Preventivos", ha='center', fontsize=11, fontweight='bold')
        plt.text(0.5, 0.92, "Pesquisa Nacional de Saúde (PNS), 2013 e 2019", ha='center', fontsize=10)
        plt.text(0.5, 0.90, "Mulheres com filhos (25 anos ou mais)", ha='center', fontsize=9, style='italic')

        # --- BLOCO PNS 2013 ---
        plt.text(0.1, 0.85, "PNS 2013", fontweight='bold', fontsize=10)
        df_2013 = tabelas['2013']['Cobertura'].reset_index()
        df_2013.columns = ['Tipo de cobertura', 'Papanicolau (%)', 'Mamografia (%)']
        
        # Criando a tabela 2013 (bbox define posição e tamanho: [esquerda, baixo, largura, altura])
        tab1 = ax.table(cellText=df_2013.values, colLabels=df_2013.columns, 
                        loc='upper center', bbox=[0.1, 0.70, 0.8, 0.14], cellLoc='center')

        # --- BLOCO PNS 2019 ---
        plt.text(0.1, 0.65, "PNS 2019", fontweight='bold', fontsize=10)
        df_2019 = tabelas['2019']['Cobertura'].reset_index()
        df_2019.columns = ['Tipo de cobertura', 'Papanicolau (%)', 'Mamografia (%)']
        
        # Criando a tabela 2019
        tab2 = ax.table(cellText=df_2019.values, colLabels=df_2019.columns, 
                        loc='upper center', bbox=[0.1, 0.54, 0.8, 0.10], cellLoc='center')

        # Estilização das Tabelas (Cores e Bordas)
        for tab in [tab1, tab2]:
            tab.auto_set_font_size(False)
            tab.set_fontsize(9)
            for (row, col), cell in tab.get_celld().items():
                # Cabeçalho: Cinza claro e Negrito
                if row == 0:
                    cell.set_text_props(weight='bold')
                    cell.set_facecolor('#E6E6E6') 
                # Alinhamento da primeira coluna à esquerda para ficar mais elegante
                if col == 0 and row > 0:
                    cell.set_text_props(ha='left')
                # Bordas bem finas
                cell.set_linewidth(0.5)

        # --- NOTA METODOLÓGICA (Corrigido o erro de alinhamento) ---
        nota_texto = (
            "Nota metodológica. Na Pesquisa Nacional de Saúde (PNS) de 2019, as variáveis que identificam "
            "explicitamente se o exame preventivo foi realizado por meio de plano de saúde não estão "
            "disponíveis no microdado, diferentemente do observado na PNS 2013. Essa ausência decorre de "
            "mudança no instrumento de coleta do IBGE, o que inviabiliza a observação direta dessa modalidade. "
            "Assim, a categoria 'plano de saúde' em 2019 é atribuída apenas quando explicitamente identificada."
        )
        
        # Quebra o texto para caber na largura da página
        linhas_nota = textwrap.fill(nota_texto, width=100)
        # Usamos ha='left' (esquerda) para evitar o erro de 'justify'
        plt.text(0.1, 0.48, linhas_nota, fontsize=8, ha='left', va='top', linespacing=1.5)

        # Rodapé final
        plt.figtext(0.1, 0.10, "Fonte: Microdados da PNS (IBGE). Elaboração própria.", fontsize=8, style='italic')

        # Salva a página no PDF
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

    print(f"O arquivo PDF foi gerado com sucesso: {filename}")

# Chamar a função
exportar_tabela4_final_pdf(tabelas)

O arquivo PDF foi gerado com sucesso: Tabela_04_Final.pdf
